In [ ]:
!pip install -q transformers accelerate peft bitsandbytes datasets trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 618.0/618.0 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 49.6 MB/s eta 0:00:00


In [ ]:
import torch
import json

from datasets import Dataset, load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)

from peft import LoraConfig, get_peft_model

In [ ]:
import torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

GPU: Tesla T4


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [ ]:
!unzip IndicLegalQA_Dataset_10K.zip

Archive:  IndicLegalQA_Dataset_10K.zip
replace IndicLegalQA Dataset_10K.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


In [ ]:
import os

os.listdir()

['.config',
 'IndicLegalQA Dataset_10K.json',
 '.ipynb_checkpoints',
 'processed_dataset',
 'IndicLegalQA_Dataset_10K.zip',
 'sample_data']

In [ ]:
import json

with open("IndicLegalQA Dataset_10K.json") as f:
    data = json.load(f)

print(type(data))
print(len(data))

<class 'list'>
10002


In [ ]:
data[0]

{'case_name': 'Union of India vs. Maj. Gen. Manomoy Ganguly',
 'judgment_date': '1st August 2018',
 'question': 'Who is the respondent in the case Union of India vs. Maj. Gen. Manomoy Ganguly?',
 'answer': 'The respondent is Maj. Gen. Manomoy Ganguly.'}

In [ ]:
def format_example(example):
    return {
        "text": f"""### Question:
{example['question']}

### Answer:
{example['answer']}"""
    }

formatted_data = [format_example(x) for x in data]

print(formatted_data[0])

{'text': '### Question:\nWho is the respondent in the case Union of India vs. Maj. Gen. Manomoy Ganguly?\n\n### Answer:\nThe respondent is Maj. Gen. Manomoy Ganguly.'}


In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(formatted_data)

print(dataset)

Dataset({
    features: ['text'],
    num_rows: 10002
})


In [ ]:
from transformers import AutoTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

In [ ]:
tokenized_dataset[0]

{'text': '### Question:\nWho is the respondent in the case Union of India vs. Maj. Gen. Manomoy Ganguly?\n\n### Answer:\nThe respondent is Maj. Gen. Manomoy Ganguly.',
 'input_ids': [1,
  835,
  894,
  29901,
  13,
  22110,
  338,
  278,
  10049,
  296,
  297,
  278,
  1206,
  7761,
  310,
  7513,
  7186,
  29889,
  6973,
  29889,
  5739,
  29889,
  2315,
  10730,
  29891,
  26531,
  11850,
  29973,
  13,
  13,
  2277,
  29937,
  673,
  29901,
  13,
  1576,
  10049,
  296,
  338,
  6973,
  29889,
  5739,
  29889,
  2315,
  10730,
  29891,
  26531,
  11850,
  29889,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2,
  2

In [ ]:
tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/10002 [00:00<?, ? examples/s]

In [ ]:
!pip install -q --upgrade transformers accelerate bitsandbytes

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
from peft import LoraConfig, get_peft_model

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.10229075496156657


In [ ]:
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.10229075496156657


In [ ]:
!pip install -q transformers==4.38.2 accelerate==0.27.2 peft==0.8.2 bitsandbytes

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.29.1 requires accelerate>=1.4.0, but you have accelerate 0.27.2 which is incompatible.
trl 0.29.1 requires transformers>=4.56.2, but you have transformers 4.38.2 which is incompatible.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.2 which is incompatible.


In [ ]:
dataset.save_to_disk("processed_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/10002 [00:00<?, ? examples/s]

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=50,
    learning_rate=2e-4,
    fp16=True,
    report_to="none"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [ ]:
trainer.train()

Step,Training Loss
10,13.382695
20,3.890045
30,0.466276
40,0.351305
50,0.319827
60,0.316400
70,0.299090
80,0.285329
90,0.286782
100,0.272464


KeyboardInterrupt: 

In [ ]:
def generate_answer(question):
    prompt = f"""### Question:
{question}

### Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
print(generate_answer("Who is the respondent in Union of India vs Manomoy Ganguly?"))

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Question:
Who is the respondent in Union of India vs Manomoy Ganguly?

### Answer:
The respondent is the Union of India and the appellant is Manomoy Ganguly.


In [ ]:
def format_cot(example):
    return {
        "text": f"""### Question:
{example['question']}

### Reasoning:
The respondent in a legal case is typically the party responding to the appeal. Based on the case title, we identify the respondent.

### Answer:
{example['answer']}"""
    }

cot_data = [format_cot(x) for x in data]

print(cot_data[0])

{'text': '### Question:\nWho is the respondent in the case Union of India vs. Maj. Gen. Manomoy Ganguly?\n\n### Reasoning:\nThe respondent in a legal case is typically the party responding to the appeal. Based on the case title, we identify the respondent.\n\n### Answer:\nThe respondent is Maj. Gen. Manomoy Ganguly.'}


In [ ]:
cot_data[0]

{'text': '### Question:\nWho is the respondent in the case Union of India vs. Maj. Gen. Manomoy Ganguly?\n\n### Reasoning:\nThe respondent in a legal case is typically the party responding to the appeal. Based on the case title, we identify the respondent.\n\n### Answer:\nThe respondent is Maj. Gen. Manomoy Ganguly.'}

In [ ]:
from datasets import Dataset

cot_dataset = Dataset.from_list(cot_data)

In [ ]:
def tokenize_cot(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

tokenized_cot = cot_dataset.map(tokenize_cot, batched=True)

Map:   0%|          | 0/10002 [00:00<?, ? examples/s]

In [ ]:
tokenized_cot[0]

{'text': '### Question:\nWho is the respondent in the case Union of India vs. Maj. Gen. Manomoy Ganguly?\n\n### Reasoning:\nThe respondent in a legal case is typically the party responding to the appeal. Based on the case title, we identify the respondent.\n\n### Answer:\nThe respondent is Maj. Gen. Manomoy Ganguly.',
 'input_ids': [1,
  835,
  894,
  29901,
  13,
  22110,
  338,
  278,
  10049,
  296,
  297,
  278,
  1206,
  7761,
  310,
  7513,
  7186,
  29889,
  6973,
  29889,
  5739,
  29889,
  2315,
  10730,
  29891,
  26531,
  11850,
  29973,
  13,
  13,
  2277,
  29937,
  830,
  1658,
  292,
  29901,
  13,
  1576,
  10049,
  296,
  297,
  263,
  11706,
  1206,
  338,
  12234,
  278,
  6263,
  10049,
  292,
  304,
  278,
  25530,
  29889,
  16564,
  373,
  278,
  1206,
  3611,
  29892,
  591,
  12439,
  278,
  10049,
  296,
  29889,
  13,
  13,
  2277,
  29937,
  673,
  29901,
  13,
  1576,
  10049,
  296,
  338,
  6973,
  29889,
  5739,
  29889,
  2315,
  10730,
  29891,
  26531

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./cot_results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    logging_steps=20,
    learning_rate=2e-4,
    fp16=True,
    report_to="none"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_cot
)

In [ ]:
trainer.train()

Step,Training Loss
20,0.370888
40,0.267712
60,0.258540
80,0.249201
100,0.240801
120,0.254464
140,0.246900
160,0.247271
180,0.237793
200,0.230526


KeyboardInterrupt: 

In [ ]:
print(generate_answer("Who is the respondent in Union of India vs Manomoy Ganguly?"))

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Question:
Who is the respondent in Union of India vs Manomoy Ganguly?

### Answer:
The respondent in Union of India vs Manomoy Ganguly is the Union of India.


In [ ]:
def generate_answer(question):
    prompt = f"""### Question:
{question}

### Reasoning:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
print(generate_answer("Who is the respondent in Union of India vs Manomoy Ganguly?"))

Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Question:
Who is the respondent in Union of India vs Manomoy Ganguly?

### Reasoning:
The respondent in a legal case is typically the party responding to the appeal. Based on the case title, we identify the respondent.

### Answer:
The respondent in the case is the Union of India, a statutory body under the Constitution of 1947, which is responsible for the administration of the National Highways in India.


In [ ]:
def format_cot_better(example):
    return {
        "text": f"""### Question:
{example['question']}

### Reasoning:
In the case "{example['case_name']}", we examine the parties involved. The respondent is the party against whom the case is filed. From the case title, we can identify that {example['answer'].replace('The respondent is ', '').replace('.', '')} is the respondent.

### Answer:
{example['answer']}"""
    }

In [ ]:
cot_data = [format_cot_better(x) for x in data]

print(cot_data[0])

{'text': '### Question:\nWho is the respondent in the case Union of India vs. Maj. Gen. Manomoy Ganguly?\n\n### Reasoning:\nIn the case "Union of India vs. Maj. Gen. Manomoy Ganguly", we examine the parties involved. The respondent is the party against whom the case is filed. From the case title, we can identify that Maj Gen Manomoy Ganguly is the respondent.\n\n### Answer:\nThe respondent is Maj. Gen. Manomoy Ganguly.'}


In [ ]:
from datasets import Dataset

cot_dataset = Dataset.from_list(cot_data)

def tokenize_cot(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_cot = cot_dataset.map(tokenize_cot, batched=True)

Map:   0%|          | 0/10002 [00:00<?, ? examples/s]

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./cot_results_v2",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    logging_steps=50,
    learning_rate=2e-4,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_cot
)

trainer.train()

Step,Training Loss
50,0.361036
100,0.297725
150,0.299521
200,0.276717
250,0.287936
300,0.294459


KeyboardInterrupt: 

In [ ]:
print(generate_answer("Who is the respondent in Union of India vs Manomoy Ganguly?"))

Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Question:
Who is the respondent in Union of India vs Manomoy Ganguly?

### Reasoning:
In the case "Union of India vs Manomoy Ganguly", we examine the parties involved. The respondent is the party against whom the case is filed. From the case title, we can identify that The respondent is the Manomoy Ganguly, an Indian citizen who claims to be a victim of sexual exploitation and abuse by Union of India and the respondent is the respondent.

### Answer:
The respondent is the Manomoy Ganguly, an Indian citizen who claims to be a victim of sexual exploitation and abuse by Union of India and the respondent.


In [ ]:
def generate_answer(question):
    prompt = f"""### Question:
{question}

### Reasoning:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,          # ❗ TURN OFF randomness
        temperature=0.0,          # ❗ deterministic
        top_p=1.0,
        repetition_penalty=1.2,   # ❗ reduce repetition
        eos_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
print(generate_answer("Who is the respondent in Union of India vs Manomoy Ganguly?"))

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Question:
Who is the respondent in Union of India vs Manomoy Ganguly?

### Reasoning:
In the case "Union of India vs. Manomoy Ganguly", we examine the parties involved. The respondent is the party against whom the case is filed. From the case title, we can identify that The respondent is the State of West Bengal, which was represented by its Commissioner for Industries and Commerce (CIC) is the respondent.

### Answer:
The respondent is the State of West Bengal, which was represented by its Commissioner for Industries and Commerce (CIC).


In [ ]:
from trl import PPOTrainer, PPOConfig

ImportError: cannot import name 'PPOTrainer' from 'trl' (/usr/local/lib/python3.12/dist-packages/trl/__init__.py)

In [ ]:
!pip install -q trl==0.7.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.9/150.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 12.5 MB/s eta 0:00:00


In [ ]:
from trl import PPOTrainer, PPOConfig

ImportError: cannot import name 'PPOTrainer' from 'trl' (/usr/local/lib/python3.12/dist-packages/trl/__init__.py)

In [ ]:
def generate_raw(question):
    prompt = f"""### Question:
{question}

### Reasoning:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
def reward_function(output, correct_answer):
    output = output.lower()
    correct = correct_answer.lower()

    score = 0

    # correct answer present
    if correct in output:
        score += 1

    # reasoning present
    if "reasoning" in output:
        score += 0.5

    # penalize hallucination keywords (basic)
    if "sexual" in output or "abuse" in output:
        score -= 1

    return score

In [ ]:
new_training_data = []

for example in data[:100]:   # small subset

    question = example["question"]
    correct_answer = example["answer"]

    output = generate_raw(question)

    reward = reward_function(output, correct_answer)

    if reward >= 1:   # only good samples
        new_training_data.append({
            "text": output
        })

Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

In [ ]:
print(len(new_training_data))

0


In [ ]:
def tokenize_rl(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_rl = rl_dataset.map(tokenize_rl)

In [ ]:
training_args = TrainingArguments(
    output_dir="./rl_results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    logging_steps=10,
    learning_rate=2e-4,
    fp16=True,
    report_to="none",
    remove_unused_columns=False   # 🔥 CRITICAL FIX
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_rl
)

In [ ]:
def reward_function(output, correct_answer):
    output = output.lower()
    correct = correct_answer.lower()

    score = 0

    # partial match (IMPORTANT)
    if any(word in output for word in correct.split()):
        score += 1

    # reasoning present
    if "reasoning" in output:
        score += 0.5

    return score

In [ ]:
from tqdm import tqdm
new_training_data = []

for example in tqdm(data[:20]):

    question = example["question"]
    correct_answer = example["answer"]

    output = generate_raw(question)

    reward = reward_function(output, correct_answer)

    if reward >= 0.5:   # 🔥 relaxed condition
        new_training_data.append({
            "text": output
        })

100%|██████████| 20/20 [03:06<00:00,  9.34s/it]


In [ ]:
print(len(new_training_data))

20


In [ ]:
rl_dataset = Dataset.from_list(new_training_data)
tokenized_rl = rl_dataset.map(tokenize_rl)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_rl
)

trainer.train()

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Step,Training Loss


TrainOutput(global_step=5, training_loss=0.24695115089416503, metrics={'train_runtime': 5.9371, 'train_samples_per_second': 3.369, 'train_steps_per_second': 0.842, 'total_flos': 63629646888960.0, 'train_loss': 0.24695115089416503, 'epoch': 1.0})

In [ ]:
print(generate_answer("Who is the respondent in Union of India vs Manomoy Ganguly?"))

Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Question:
Who is the respondent in Union of India vs Manomoy Ganguly?

### Reasoning:
In the case "Union of India vs. Manomoy Ganguly", we examine the parties involved. The respondent is the party against whom the case is filed. From the case title, we can identify that The respondent is the State of West Bengal, which was represented by its Commissioner for Industries and Commerce, Kolkata, and the Director General of Inspection (DGI) is the respondent.


In [ ]:
def generate_answer(question):
    prompt = f"""### Question:
{question}

### Reasoning:
Identify the parties in the case title. The respondent is usually the second party mentioned.

### Answer:
The respondent is"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=20,        # 🔥 VERY IMPORTANT (limit drift)
        do_sample=False,
        repetition_penalty=1.2,
        eos_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
print(generate_answer("Who is the respondent in Union of India vs Manomoy Ganguly?"))


Both `max_new_tokens` (=20) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Question:
Who is the respondent in Union of India vs Manomoy Ganguly?

### Reasoning:
Identify the parties in the case title. The respondent is usually the second party mentioned.

### Answer:
The respondent is the State of West Bengal, represented by its Commissioner and Director General (Revenue), Kol


In [ ]:
def extract_respondent(question):
    if "vs" in question.lower():
        parts = question.split("vs")
        return parts[1].strip().replace("?", "")
    return None

In [ ]:
def final_answer(question):

    respondent = extract_respondent(question)

    prompt = f"""### Question:
{question}

### Reasoning:
In the case title, the respondent is typically the second party mentioned, which is {respondent}.

### Answer:
The respondent is {respondent}.
"""

    return prompt

In [ ]:
print(final_answer("Who is the respondent in Union of India vs Manomoy Ganguly?"))

### Question:
Who is the respondent in Union of India vs Manomoy Ganguly?

### Reasoning:
In the case title, the respondent is typically the second party mentioned, which is Manomoy Ganguly.

### Answer:
The respondent is Manomoy Ganguly.



In [ ]:
def compute_accuracy(data, model_fn):
    correct = 0

    for example in data[:50]:
        question = example["question"]
        true_answer = example["answer"].lower()

        pred = model_fn(question).lower()

        if true_answer in pred:
            correct += 1

    return correct / 50

In [ ]:
def exact_match(data, model_fn):
    correct = 0

    for example in data[:50]:
        pred = model_fn(example["question"]).strip().lower()
        true = example["answer"].strip().lower()

        if pred == true:
            correct += 1

    return correct / 50

In [ ]:
def reasoning_score(data, model_fn):
    count = 0

    for example in data[:50]:
        output = model_fn(example["question"]).lower()

        if "reasoning" in output:
            count += 1

    return count / 50

In [ ]:
def hallucination_rate(data, model_fn):
    hallucinations = 0

    for example in data[:50]:
        output = model_fn(example["question"]).lower()
        answer = example["answer"].lower()

        if answer not in output:
            hallucinations += 1

    return hallucinations / 50

In [ ]:
compute_accuracy(data)
exact_match(data)
reasoning_score(data)
hallucination_rate(data)

TypeError: compute_accuracy() missing 1 required positional argument: 'model_fn'